In [19]:
import pandas as pd
import numpy as np

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [20]:
master = pd.read_csv(
    'cleaned_master_dataset.csv'
)

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [21]:
master.shape

(82365, 39)

# Convert Purchase Timestamp

In [22]:
master['order_purchase_timestamp'] = pd.to_datetime(
    master['order_purchase_timestamp'],
    format='mixed',
    dayfirst=True
)

master['order_purchase_timestamp'].head()

,order_purchase_timestamp
0,2017-10-02 10:56:00
1,2017-10-02 10:56:00
2,2017-10-02 10:56:00
3,2018-07-24 20:41:00
4,2018-08-08 08:38:00


# Create Snapshot Date

In [23]:
snapshot_date = (
    master['order_purchase_timestamp']
    .max()
    +
    pd.Timedelta(days=1)
)

snapshot_date

Timestamp('2018-10-18 17:30:00')

# Calculate Customer Recency

In [24]:
customer_recency = (
    master
    .groupby(
        'customer_unique_id'
    )['order_purchase_timestamp']
    .max()
    .reset_index()
)

customer_recency['Recency'] = (
    snapshot_date
    -
    customer_recency[
        'order_purchase_timestamp'
    ]
).dt.days

customer_recency.head()

,customer_unique_id,order_purchase_timestamp,Recency
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:00,161
1,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:00,586
2,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:00,337
3,00053a61a98854899e70ed204dd4bafe,2018-02-28 11:15:00,232
4,0005ef4cd20d2893f0d9fbd94d3c0d97,2018-03-12 15:22:00,220


# Create Churn Flag

In [25]:
customer_recency['Churn'] = np.where(
    customer_recency['Recency'] > 90,
    1,
    0
)

customer_recency.head()

,customer_unique_id,order_purchase_timestamp,Recency,Churn
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:00,161,1
1,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:00,586,1
2,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:00,337,1
3,00053a61a98854899e70ed204dd4bafe,2018-02-28 11:15:00,232,1
4,0005ef4cd20d2893f0d9fbd94d3c0d97,2018-03-12 15:22:00,220,1


# Calculate Churn Rate

In [26]:
churn_rate = (
    customer_recency['Churn']
    .mean()
    * 100
)

print(
    "Churn Rate:",
    round(churn_rate,2),
    "%"
)

Churn Rate: 90.13 %


# Customer Risk Segmentation

In [27]:
def risk_level(days):

    if days > 180:
        return 'Critical Risk'

    elif days > 120:
        return 'High Risk'

    elif days > 60:
        return 'Medium Risk'

    else:
        return 'Low Risk'


customer_recency[
    'Risk_Level'
] = customer_recency[
    'Recency'
].apply(
    risk_level
)

customer_recency[
    'Risk_Level'
].value_counts()

,count
Risk_Level,
Critical Risk,49162
Medium Risk,9657
High Risk,9219
Low Risk,1089


# Revenue at Risk Analysis

In [28]:
customer_revenue = (
    master
    .groupby(
        'customer_unique_id'
    )['revenue']
    .sum()
    .reset_index()
)

customer_churn = pd.merge(
    customer_recency,
    customer_revenue,
    on='customer_unique_id'
)

customer_churn.head()

,customer_unique_id,order_purchase_timestamp,Recency,Churn,Risk_Level,revenue
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:00,161,1,High Risk,0.0
1,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:00,586,1,Critical Risk,69.0
2,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:00,337,1,Critical Risk,0.0
3,00053a61a98854899e70ed204dd4bafe,2018-02-28 11:15:00,232,1,Critical Risk,382.0
4,0005ef4cd20d2893f0d9fbd94d3c0d97,2018-03-12 15:22:00,220,1,Critical Risk,104.9


# Revenue by Risk Segment

In [29]:
risk_revenue = (
    customer_churn
    .groupby(
        'Risk_Level'
    )['revenue']
    .sum()
    .reset_index()
    .sort_values(
        'revenue',
        ascending=False
    )
)

risk_revenue

,Risk_Level,revenue
0,Critical Risk,5799109.75
3,Medium Risk,1179039.80
1,High Risk,1138102.48
2,Low Risk,103492.34


# Customer Health Score

In [30]:
customer_churn['Health_Score'] = (
    100
    -
    (
        customer_churn['Recency']
        /
        customer_churn['Recency'].max()
        * 100
    )
)

customer_churn[
    'Health_Score'
] = customer_churn[
    'Health_Score'
].round(2)

customer_churn[
    [
        'customer_unique_id',
        'Recency',
        'Health_Score'
    ]
].head()

,customer_unique_id,Recency,Health_Score
0,0000366f3b9a7992bf8c76cfdf3221e2,161,79.17
1,0000f46a3911fa3c0805444483337064,586,24.19
2,0004aac84e0df4da2b147fca70cf8255,337,56.40
3,00053a61a98854899e70ed204dd4bafe,232,69.99
4,0005ef4cd20d2893f0d9fbd94d3c0d97,220,71.54


# Create Health Category

In [31]:
def health_category(score):

    if score >= 75:
        return 'Excellent'

    elif score >= 50:
        return 'Good'

    elif score >= 25:
        return 'Average'

    else:
        return 'Poor'


customer_churn[
    'Health_Category'
] = customer_churn[
    'Health_Score'
].apply(
    health_category
)

customer_churn[
    'Health_Category'
].value_counts()

,count
Health_Category,
Good,28585
Excellent,22072
Average,15501
Poor,2969


# Top At-Risk Customers

In [32]:
top_risk_customers = (
    customer_churn
    .sort_values(
        'Recency',
        ascending=False
    )
    .head(100)
)

top_risk_customers.head(10)

,customer_unique_id,order_purchase_timestamp,Recency,Churn,Risk_Level,revenue,Health_Score,Health_Category
19518,4854e9b3feff728c13ee5fc7d1547e92,2016-09-05 00:15:00,773,1,Critical Risk,0.00,0.00,Poor
49647,b7d76e111c89f7ebf14761390f0f7d17,2016-09-04 21:15:00,773,1,Critical Risk,72.89,0.00,Poor
175,009b0127b727ab0ba422f6d9604487c7,2016-09-13 15:24:00,765,1,Critical Risk,0.00,1.03,Poor
35363,830d5b7aaa3b6f1e9ad63703bec97d23,2016-09-15 12:16:00,763,1,Critical Risk,134.97,1.29,Poor
12813,2f64e403852e6893ae37485d5fcacdaf,2016-10-03 16:56:00,745,1,Critical Risk,21.90,3.62,Poor
61412,e3299196e1482e1ae1320fcb594bc23e,2016-10-04 15:10:00,744,1,Critical Risk,89.90,3.75,Poor
13154,30a38716bb2d04f12bb813aa2f926270,2016-10-04 13:30:00,744,1,Critical Risk,0.00,3.75,Poor
14977,37751f6b47c9a1dc55fffde08f0183fc,2016-10-04 16:05:00,744,1,Critical Risk,0.00,3.75,Poor
29798,6e4c8eed6842146f77d917f7b437b4c2,2016-10-04 11:03:00,744,1,Critical Risk,148.00,3.75,Poor
4728,1151be76113653dc4ed5d7e7fd17763a,2016-10-04 16:40:00,744,1,Critical Risk,59.90,3.75,Poor


# Customer Action Recommendation Engine

In [33]:
def customer_action(risk):

    if risk == 'Critical Risk':
        return 'Win-Back Campaign'

    elif risk == 'High Risk':
        return 'Retention Offer'

    elif risk == 'Medium Risk':
        return 'Promotional Email'

    else:
        return 'Maintain Engagement'


customer_churn['Recommended_Action'] = (
    customer_churn['Risk_Level']
    .apply(customer_action)
)

customer_churn[
    [
        'customer_unique_id',
        'Risk_Level',
        'Recommended_Action'
    ]
].head()

,customer_unique_id,Risk_Level,Recommended_Action
0,0000366f3b9a7992bf8c76cfdf3221e2,High Risk,Retention Offer
1,0000f46a3911fa3c0805444483337064,Critical Risk,Win-Back Campaign
2,0004aac84e0df4da2b147fca70cf8255,Critical Risk,Win-Back Campaign
3,00053a61a98854899e70ed204dd4bafe,Critical Risk,Win-Back Campaign
4,0005ef4cd20d2893f0d9fbd94d3c0d97,Critical Risk,Win-Back Campaign


# Recommended Action Distribution

In [34]:
customer_churn[
    'Recommended_Action'
].value_counts()

,count
Recommended_Action,
Win-Back Campaign,49162
Promotional Email,9657
Retention Offer,9219
Maintain Engagement,1089


# Create Dashboard Dataset

In [35]:
dashboard_dataset = customer_churn[
    [
        'customer_unique_id',
        'Recency',
        'Churn',
        'Risk_Level',
        'Health_Score',
        'Health_Category',
        'Recommended_Action',
        'revenue'
    ]
]

dashboard_dataset.head()

,customer_unique_id,Recency,Churn,Risk_Level,Health_Score,Health_Category,Recommended_Action,revenue
0,0000366f3b9a7992bf8c76cfdf3221e2,161,1,High Risk,79.17,Excellent,Retention Offer,0.0
1,0000f46a3911fa3c0805444483337064,586,1,Critical Risk,24.19,Poor,Win-Back Campaign,69.0
2,0004aac84e0df4da2b147fca70cf8255,337,1,Critical Risk,56.40,Good,Win-Back Campaign,0.0
3,00053a61a98854899e70ed204dd4bafe,232,1,Critical Risk,69.99,Good,Win-Back Campaign,382.0
4,0005ef4cd20d2893f0d9fbd94d3c0d97,220,1,Critical Risk,71.54,Good,Win-Back Campaign,104.9


In [36]:
customer_churn.to_csv(
    'customer_churn.csv',
    index=False
)

print("Customer Churn Dataset Saved")

Customer Churn Dataset Saved


In [37]:
dashboard_dataset.to_csv(
    'churn_dashboard.csv',
    index=False
)

print("Dashboard Dataset Saved")

Dashboard Dataset Saved


In [38]:
risk_revenue.to_csv(
    'risk_revenue.csv',
    index=False
)

print("Risk Revenue Dataset Saved")

Risk Revenue Dataset Saved


In [39]:
import os

os.listdir()

['.config',
 'cleaned_master_dataset.csv',
 'risk_revenue.csv',
 'churn_dashboard.csv',
 'customer_churn.csv',
 'sample_data']

In [40]:
customer_churn['Risk_Score'] = (
    customer_churn['Recency']
    /
    customer_churn['Recency'].max()
    * 100
).round(2)

customer_churn[
    [
        'customer_unique_id',
        'Risk_Score'
    ]
].head()

,customer_unique_id,Risk_Score
0,0000366f3b9a7992bf8c76cfdf3221e2,20.83
1,0000f46a3911fa3c0805444483337064,75.81
2,0004aac84e0df4da2b147fca70cf8255,43.60
3,00053a61a98854899e70ed204dd4bafe,30.01
4,0005ef4cd20d2893f0d9fbd94d3c0d97,28.46
